# Airbnb Price Prediction

Prédiction du **logarithme du prix de nuit** pour des logements Airbnb.  
Rendu : fichier `predictions.csv` au format `(id, logpred)`.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import mean_squared_error
import re
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

try:
    import lightgbm as lgb
    USE_LGBM = True
    print(f"LightGBM {lgb.__version__} disponible")
except ImportError:
    USE_LGBM = False
    print("LightGBM non disponible — fallback GradientBoostingRegressor")

plt.rcParams['figure.figsize'] = (12, 5)
pd.set_option('display.max_columns', 50)
SEED = 42
np.random.seed(SEED)

## 1. Chargement des données

In [ ]:
train = pd.read_csv('airbnb_train.csv')
test  = pd.read_csv('airbnb_test.csv')

# La 1ère colonne du test est sans nom (c'est l'id)
test.rename(columns={test.columns[0]: 'id'}, inplace=True)

print(f"Train : {train.shape[0]:,} lignes x {train.shape[1]} colonnes")
print(f"Test  : {test.shape[0]:,} lignes x {test.shape[1]} colonnes")
train.head(3)

## 2. Exploration des données (EDA)

In [ ]:
# Valeurs manquantes
def missing_report(df, name):
    m = (df.isnull() | (df == '')).sum()
    m = m[m > 0].sort_values(ascending=False)
    pct = (m / len(df) * 100).round(1)
    print(f"\n--- {name} ---")
    print(pd.DataFrame({'manquants': m, '%': pct}).to_string())

missing_report(train, 'Train')

In [ ]:
# Distribution de la variable cible
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(train['log_price'], bins=60, color='steelblue', edgecolor='white', lw=0.5)
axes[0].axvline(train['log_price'].mean(), color='red', ls='--',
                label=f"Moyenne : {train['log_price'].mean():.2f}")
axes[0].set_title('Distribution de log_price'); axes[0].set_xlabel('log_price')
axes[0].legend()

axes[1].hist(np.exp(train['log_price']), bins=100, color='coral', edgecolor='white', lw=0.5)
axes[1].set_xlim(0, 800); axes[1].set_title('Distribution du prix ($)')
axes[1].set_xlabel('Prix ($)')

plt.suptitle('Variable cible', fontsize=14, y=1.02)
plt.tight_layout(); plt.show()
print(f"min={train['log_price'].min():.2f}  max={train['log_price'].max():.2f}  "
      f"mean={train['log_price'].mean():.2f}  std={train['log_price'].std():.2f}")

In [ ]:
# Prix moyen par catégorie
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flatten(), ['room_type','property_type','city','cancellation_policy']):
    means = train.groupby(col)['log_price'].mean().sort_values(ascending=False)
    colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(means)))
    bars = ax.bar(range(len(means)), means.values, color=colors, edgecolor='white')
    ax.set_xticks(range(len(means)))
    ax.set_xticklabels(means.index, rotation=45, ha='right', fontsize=9)
    ax.set_title(f'log_price moyen par {col}', fontsize=11)
    for bar, val in zip(bars, means.values):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height()+0.02,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
plt.tight_layout(); plt.show()

In [ ]:
# Corrélations numériques
num_cols = ['accommodates','bathrooms','bedrooms','beds','number_of_reviews','review_scores_rating']
tmp = train[num_cols + ['log_price']].copy()
for c in num_cols:
    tmp[c] = pd.to_numeric(tmp[c], errors='coerce')
corr = tmp.corr()['log_price'].drop('log_price').sort_values(key=abs, ascending=False)
print("Corrélations avec log_price:")
print(corr.round(3))

## 3. Feature Engineering

- **Amenities** : nombre total + 30 amenities les plus fréquentes (binaires)
- **Dates** : `host_since`, `first_review`, `last_review` → jours jusqu'au 01/01/2018
- **Booleans** : `t/f` / `True/False` → 0/1
- **Texte** : longueur et nombre de mots de la description, longueur du nom
- **Taux de réponse** : `"90%"` → `0.9`

In [ ]:
def parse_amenities(s):
    if pd.isna(s) or s == '':
        return []
    items = []
    for item in str(s).strip('{}').split(','):
        item = item.strip().strip('"').strip()
        if item:
            items.append(item)
    return items

# Top 30 amenities (calculé sur le train uniquement)
all_am = []
for a in train['amenities']:
    all_am.extend(parse_amenities(a))
am_counts = Counter(all_am)
TOP_AMENITIES = [a for a, _ in am_counts.most_common(30)]

print("Top 30 amenities :")
for i, (a, c) in enumerate(am_counts.most_common(30), 1):
    print(f"  {i:2d}. {a:<45} ({c:,})")

In [ ]:
def engineer_features(df):
    df = df.copy()

    # Amenities
    parsed_am = df['amenities'].apply(parse_amenities)
    df['amenities_count'] = parsed_am.apply(len)
    for am in TOP_AMENITIES:
        col = 'am_' + re.sub(r'[^a-z0-9]', '_', am.lower())[:30]
        df[col] = parsed_am.apply(lambda x: 1 if am in x else 0)

    # Dates
    ref = pd.Timestamp('2018-01-01')
    for dc in ['host_since', 'first_review', 'last_review']:
        parsed = pd.to_datetime(df[dc], errors='coerce')
        df[dc + '_days']    = (ref - parsed).dt.days
        df[dc + '_missing'] = parsed.isna().astype(int)

    # Booleans
    for c in ['host_has_profile_pic', 'host_identity_verified', 'instant_bookable']:
        df[c] = (df[c].astype(str).str.lower() == 't').astype(int)
    df['cleaning_fee'] = (df['cleaning_fee'].astype(str).str.lower() == 'true').astype(int)

    # Taux de réponse
    def parse_rate(x):
        try:
            return float(str(x).replace('%','').strip()) / 100.0
        except Exception:
            return np.nan
    df['host_response_rate'] = df['host_response_rate'].apply(
        lambda x: np.nan if pd.isna(x) or str(x).strip() == '' else parse_rate(x))

    # Texte
    df['description_len']   = df['description'].fillna('').apply(len)
    df['description_words'] = df['description'].fillna('').apply(lambda x: len(x.split()))
    df['name_len']           = df['name'].fillna('').apply(len)

    # Numériques
    for c in ['bathrooms','bedrooms','beds','review_scores_rating','number_of_reviews','accommodates']:
        df[c] = pd.to_numeric(df[c], errors='coerce')

    # Indicateur manquant avis
    df['review_missing'] = df['review_scores_rating'].isna().astype(int)

    # Nettoyage haute cardinalité
    df['zipcode_clean']       = df['zipcode'].fillna('Unknown').astype(str).str.strip().str[:5]
    df['neighbourhood_clean'] = df['neighbourhood'].fillna('Unknown').astype(str).str.strip()

    return df

train_fe = engineer_features(train)
test_fe  = engineer_features(test)
print(f"Colonnes après feature engineering : {train_fe.shape[1]}")

## 4. Encodage des variables catégorielles

- **Target encoding** (neighbourhood, zipcode) : moyenne de `log_price` avec cross-validation 5-fold pour éviter le data leakage
- **Label encoding** (property_type, room_type, bed_type, cancellation_policy, city)

In [ ]:
def target_encode(train_df, test_df, cols, target='log_price', n_splits=5, smoothing=10):
    global_mean = train_df[target].mean()
    for col in cols:
        enc_tr = np.full(len(train_df), global_mean, dtype=float)
        kf = KFold(n_splits=n_splits, shuffle=True, random_state=SEED)
        for tr_idx, val_idx in kf.split(train_df):
            fold_tr  = train_df.iloc[tr_idx]
            fold_val = train_df.iloc[val_idx]
            stats  = fold_tr.groupby(col)[target].agg(['mean','count'])
            smooth = (stats['count']*stats['mean'] + smoothing*global_mean) / (stats['count']+smoothing)
            enc_tr[val_idx] = fold_val[col].map(smooth).fillna(global_mean).values
        stats_all  = train_df.groupby(col)[target].agg(['mean','count'])
        smooth_all = (stats_all['count']*stats_all['mean'] + smoothing*global_mean) / (stats_all['count']+smoothing)
        train_df[col+'_te'] = enc_tr
        test_df[col+'_te']  = test_df[col].map(smooth_all).fillna(global_mean).values
    return train_df, test_df

def label_encode(train_df, test_df, cols):
    for col in cols:
        le = LabelEncoder()
        combined = pd.concat([train_df[col].fillna('Unknown').astype(str),
                               test_df[col].fillna('Unknown').astype(str)], ignore_index=True)
        le.fit(combined)
        train_df[col+'_le'] = le.transform(train_df[col].fillna('Unknown').astype(str))
        test_df[col+'_le']  = le.transform(test_df[col].fillna('Unknown').astype(str))
    return train_df, test_df

TE_COLS = ['neighbourhood_clean', 'zipcode_clean']
LE_COLS = ['property_type', 'room_type', 'bed_type', 'cancellation_policy', 'city']

train_enc, test_enc = target_encode(train_fe, test_fe, TE_COLS)
train_enc, test_enc = label_encode(train_enc, test_enc, LE_COLS)
print("Encodage terminé")

In [ ]:
NUM_FEAT = [
    'accommodates','bathrooms','bedrooms','beds',
    'number_of_reviews','review_scores_rating','latitude','longitude',
    'amenities_count','description_len','description_words','name_len',
    'host_response_rate',
    'host_since_days','first_review_days','last_review_days',
    'host_since_missing','first_review_missing','last_review_missing',
    'review_missing',
    'host_has_profile_pic','host_identity_verified','instant_bookable','cleaning_fee',
]
TE_FEAT  = [c+'_te' for c in TE_COLS]
LE_FEAT  = [c+'_le' for c in LE_COLS]
AM_FEAT  = [c for c in train_enc.columns if c.startswith('am_')]
ALL_FEAT = NUM_FEAT + TE_FEAT + LE_FEAT + AM_FEAT

print(f"Total features : {len(ALL_FEAT)}")
print(f"  Numériques     : {len(NUM_FEAT)}")
print(f"  Target encoded : {len(TE_FEAT)}")
print(f"  Label encoded  : {len(LE_FEAT)}")
print(f"  Amenities      : {len(AM_FEAT)}")

In [ ]:
X      = train_enc[ALL_FEAT].copy()
y      = train_enc['log_price'].copy()
X_test = test_enc[ALL_FEAT].copy()

medians = X.median()
X       = X.fillna(medians)
X_test  = X_test.fillna(medians)

print(f"X      : {X.shape}")
print(f"y      : {y.shape}  mean={y.mean():.3f}  std={y.std():.3f}")
print(f"X_test : {X_test.shape}")
print(f"NaN dans X      : {X.isna().sum().sum()}")
print(f"NaN dans X_test : {X_test.isna().sum().sum()}")

## 5. Modélisation

**LightGBM** avec validation croisée 5-fold et early stopping pour trouver le bon nombre d'itérations.  
Fallback vers `GradientBoostingRegressor` si LightGBM n'est pas installé.  
Métrique : **RMSE sur log_price**.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds = np.zeros(len(X))
cv_scores = []

if USE_LGBM:
    params = dict(
        objective='regression', metric='rmse',
        learning_rate=0.05, num_leaves=63, max_depth=-1,
        min_child_samples=20, feature_fraction=0.8,
        bagging_fraction=0.8, bagging_freq=5,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=SEED, verbose=-1, n_jobs=-1,
    )
    best_iters = []

    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        dtrain = lgb.Dataset(X_tr, label=y_tr, feature_name=ALL_FEAT)
        dval   = lgb.Dataset(X_val, label=y_val)
        model  = lgb.train(params, dtrain, num_boost_round=3000,
                           valid_sets=[dval],
                           callbacks=[lgb.early_stopping(150, verbose=False),
                                      lgb.log_evaluation(500)])
        preds = model.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse); best_iters.append(model.best_iteration)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f} | best_iter={model.best_iteration}")

    best_n = int(np.mean(best_iters) * 1.05)
    print(f"\nMoyenne best_iter={int(np.mean(best_iters))} → modèle final : {best_n} rounds")

else:
    from sklearn.ensemble import GradientBoostingRegressor
    best_n = 300
    for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]
        m = GradientBoostingRegressor(n_estimators=300, learning_rate=0.1,
                                       max_depth=5, subsample=0.8, random_state=SEED)
        m.fit(X_tr, y_tr); preds = m.predict(X_val)
        oof_preds[val_idx] = preds
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        cv_scores.append(rmse)
        print(f"  Fold {fold+1}/5 | RMSE={rmse:.4f}")

print(f"\n{'='*40}")
print(f"CV RMSE : {np.mean(cv_scores):.4f} ± {np.std(cv_scores):.4f}")
print(f"{'='*40}")

In [ ]:
# Modèle final entraîné sur tout le train
print(f"Entraînement final ({best_n} itérations)...")

if USE_LGBM:
    dtrain_full = lgb.Dataset(X, label=y, feature_name=ALL_FEAT)
    final_model = lgb.train(params, dtrain_full, num_boost_round=best_n)
    test_preds  = final_model.predict(X_test)
else:
    final_model = GradientBoostingRegressor(n_estimators=best_n, learning_rate=0.1,
                                             max_depth=5, subsample=0.8, random_state=SEED)
    final_model.fit(X, y)
    test_preds = final_model.predict(X_test)

print(f"Prédictions test — min={test_preds.min():.3f}  max={test_preds.max():.3f}  "
      f"mean={test_preds.mean():.3f}")

## 6. Analyse du modèle

In [ ]:
# Feature importance
if USE_LGBM:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importance(importance_type='gain')})
else:
    imp = pd.DataFrame({'feature': ALL_FEAT,
                        'importance': final_model.feature_importances_})

imp = imp.sort_values('importance', ascending=False).reset_index(drop=True)
top20 = imp.head(20)

fig, ax = plt.subplots(figsize=(10, 8))
colors = plt.cm.viridis(np.linspace(0.2, 0.8, len(top20)))
ax.barh(range(len(top20)), top20['importance'].values, color=colors)
ax.set_yticks(range(len(top20))); ax.set_yticklabels(top20['feature'].values, fontsize=10)
ax.invert_yaxis(); ax.set_title('Top 20 features (importance gain)', fontsize=13)
plt.tight_layout(); plt.show()

print("Top 10 features:")
print(imp.head(10).to_string(index=False))

In [ ]:
# OOF : prédictions vs réel + distribution des résidus
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].scatter(y, oof_preds, alpha=0.1, s=1, color='steelblue')
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Parfait')
axes[0].set_xlabel('log_price réel'); axes[0].set_ylabel('log_price prédit (OOF)')
axes[0].set_title('OOF : prédit vs réel'); axes[0].legend()

residuals = y - oof_preds
axes[1].hist(residuals, bins=60, color='coral', edgecolor='white', lw=0.5)
axes[1].axvline(0, color='black', ls='--')
axes[1].set_title(f'Résidus (std={residuals.std():.3f})')
axes[1].set_xlabel('Résidu (réel − prédit)')

plt.tight_layout(); plt.show()
oof_rmse = np.sqrt(mean_squared_error(y, oof_preds))
print(f"OOF RMSE global : {oof_rmse:.4f}")

## 7. Export des prédictions

In [ ]:
submission = pd.DataFrame({'logpred': test_preds}, index=test_enc['id'].values)
submission.index.name = ''
submission.to_csv('predictions.csv')
print(f"Sauvegardé : predictions.csv  ({len(submission):,} lignes)")
submission.head()

In [ ]:
# Vérification du format
example = pd.read_csv('prediction_example.csv', index_col=0)
pred    = pd.read_csv('predictions.csv', index_col=0)

print("=== Vérification ===")
print(f"Colonnes attendues : {list(example.columns)}")
print(f"Colonnes obtenues  : {list(pred.columns)}")
print(f"Lignes             : {len(pred):,}")
print()
print(pred.head(5))
print()

plt.figure(figsize=(10, 4))
plt.hist(pred['logpred'], bins=60, color='steelblue', edgecolor='white', lw=0.5, label='Test (prédit)')
plt.axvline(y.mean(), color='red', ls='--', label=f'Moyenne train ({y.mean():.2f})')
plt.title('Distribution des prédictions'); plt.xlabel('log_price prédit')
plt.legend(); plt.tight_layout(); plt.show()
print("Export OK!")